# Xu Lab BindCraft2 on CMRI HPC - binder design notebook

Adapted from the official `bindcraft.ipynb` for **regular Jupyter on the CMRI HPC**.
It keeps the upstream notebook's shape (set up, targets, binder, run, results, structures,
download) and replaces the Google Colab parts:

| Upstream Colab notebook | This notebook |
| --- | --- |
| mounts Google Drive | reads and writes `/project/yli4/ypan/bindcraft/` |
| `pip install`s into the runtime | uses the existing `bindcraft2` conda environment; installs nothing |
| `google.colab.output.enable_custom_widget_manager()` | plain `ipywidgets`, no Colab imports |
| runs the campaign **inside** the notebook process | submits a **SLURM batch job** to an H200 |
| `files.download(...)` | writes an export folder plus a zip, with Jupyter file links |
| downloads AlphaFold weights on demand | reuses the parameter set already on the cluster |

**What this notebook does not do:** it never designs anything in the kernel. Every campaign is a
batch job, so you can close the notebook, log out, and come back to the same campaign. The
kernel also never claims GPU success - the H200 is only ever evidenced by the job's own log, which
the launcher writes and `validate_outputs.py` reads.

**Run order:** cell 1 once per session, then 2-4 to submit, then 5-9 for any campaign (past or
present). Cells 5-9 work on a campaign that is still running.

**Filters stay on.** The Colab demo's `no_filters` shortcut is deliberately not offered here: a
campaign without acceptance filters produces numbers that mean nothing.

## 1 - Set up and check the environment

In [ ]:
# Imports the helper layer beside this notebook and reports what is actually present.
# Nothing is installed and no GPU claim is made: a Jupyter node is not the batch allocation.
import os, sys, json, importlib
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
for candidate in (NOTEBOOK_DIR, NOTEBOOK_DIR / "lab" / "notebooks",
                  Path("/project/yli4/ypan/bindcraft/BindCraft2/lab/notebooks")):
    if (candidate / "xulab_hpc.py").is_file():
        sys.path.insert(0, str(candidate))
        break

import xulab_hpc as hpc
importlib.reload(hpc)

print(f"source tree   : {hpc.SOURCE_TREE}")
print(f"results root  : {hpc.RESULTS_ROOT}")
print(f"logs root     : {hpc.LOGS_ROOT}")
print(f"interpreter   : {hpc.CONDA_PYTHON}")
print(f"launcher      : {hpc.SLURM_SCRIPT}")
print()

ENVIRONMENT = hpc.environment_report()
width = max(len(name) for name, _, _ in ENVIRONMENT)
for name, status, detail in ENVIRONMENT:
    print(f"{status:11s} {name:<{width}}  {detail}")

status, detail = hpc.imported_bindcraft_path()
print(f"\n{status:11s} bindcraft import origin  {detail}")

BLOCKING = [name for name, state, _ in ENVIRONMENT if state == "FAIL"]
if BLOCKING or status == "FAIL":
    print("\nBLOCKED: fix the FAIL lines before submitting. Missing directories are created with")
    print(f"  mkdir -p {hpc.PROJECT_ROOT}/{{results,logs,cache/jax,tmp}}")
    print("and the full checklist is lab/scripts/hpc_preflight.sh on a login node.")
else:
    print("\nREADY to configure a campaign (GPU use is still unproven until a job runs).")

## 2 - Target

Pick a shipped target or point at your own PDB / mmCIF / FASTA file **already on the cluster**.
`hotspots` are the residues to bind, `coldspots` the residues to keep clear, in the numbering of
the target file. Add as many targets as the campaign needs; several targets make it a
multi-target campaign.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

STYLE = {"description_width": "180px"}
def wide(width="560px"):
    return widgets.Layout(width=width)   # a Layout belongs to one widget, so build a new one

TARGETS = []          # the campaign's target entries, in the order added

source = widgets.Dropdown(options=["a shipped target", "a file on the cluster"],
                          description="target from", style=STYLE, layout=wide())
shipped = widgets.Dropdown(options=hpc.available_targets(),
                           value="hPDL1" if "hPDL1" in hpc.available_targets() else None,
                           description="which one", style=STYLE, layout=wide())
file_path = widgets.Text(description="file path",
                         placeholder="/project/yli4/ypan/bindcraft/targets/my_target.pdb",
                         style=STYLE, layout=wide("720px"))
name = widgets.Text(description="name (optional)", style=STYLE, layout=wide())
chains = widgets.Text(description="target chains", placeholder="A  or  A,B",
                      style=STYLE, layout=wide())
hotspots = widgets.Text(description="hotspots (bind these)", placeholder="54,56,66,115",
                        style=STYLE, layout=wide())
coldspots = widgets.Text(description="coldspots (avoid these)", placeholder="90-95",
                         style=STYLE, layout=wide())
add = widgets.Button(description="add this target", button_style="primary",
                     layout=widgets.Layout(width="180px"))
clear = widgets.Button(description="remove all", layout=widgets.Layout(width="140px"))
chosen = widgets.Output()

def on_source(change=None):
    shipped.layout.display = None if source.value == "a shipped target" else "none"
    file_path.layout.display = "none" if source.value == "a shipped target" else None

def show_targets():
    with chosen:
        chosen.clear_output(wait=True)
        if not TARGETS:
            print("no target yet")
            return
        for index, target in enumerate(TARGETS, start=1):
            print(f"{index}. {json.dumps(target)}")

def on_add(_=None):
    with chosen:
        try:
            if source.value == "a shipped target":
                entry = hpc.preset_target(shipped.value)
                # Anything typed here overrides what the shipped preset carries.
                for key, box in (("name", name), ("chains", chains),
                                 ("hotspots", hotspots), ("coldspots", coldspots)):
                    if box.value.strip():
                        entry[key] = box.value.strip()
            else:
                entry = hpc.custom_target(file_path.value, name.value, chains.value,
                                          hotspots.value, coldspots.value)
        except Exception as failure:
            chosen.clear_output(wait=True)
            print(f"not added: {type(failure).__name__}: {failure}")
            return
        TARGETS.append(entry)
    show_targets()

def on_clear(_=None):
    TARGETS.clear()
    show_targets()

source.observe(on_source, names="value")
add.on_click(on_add)
clear.on_click(on_clear)
on_source()
display(widgets.VBox([source, shipped, file_path, name, chains, hotspots, coldspots,
                      widgets.HBox([add, clear]), chosen]))
show_targets()

## 3 - Binder and campaign settings

`max_trajectories` is the work budget; `number_of_final_designs` is when the campaign stops
early because it has enough. A short campaign can legitimately accept nothing.
Scaffolded types (VHH, ARP, scFv, Fab) bring their own lengths, so the length boxes are ignored
for them.

In [ ]:
campaign_name = widgets.Text(value="pdl1_h200_smoke", description="campaign name",
                             style=STYLE, layout=wide())
binder_type = widgets.Dropdown(options=list(hpc.BINDER_TYPES), value="de novo binder",
                               description="binder type", style=STYLE, layout=wide())
shortest = widgets.BoundedIntText(value=60, min=1, max=2000, description="shortest binder",
                                  style=STYLE, layout=wide("340px"))
longest = widgets.BoundedIntText(value=100, min=1, max=2000, description="longest binder",
                                 style=STYLE, layout=wide("340px"))
copies = widgets.BoundedIntText(value=1, min=1, max=12, description="copies (oligomer)",
                                style=STYLE, layout=wide("340px"))
properties = widgets.SelectMultiple(options=list(hpc.PROPERTY_OPTIONS), rows=8,
                                    description="extra properties", style=STYLE, layout=wide())
final_designs = widgets.BoundedIntText(value=1, min=1, max=500, description="final designs wanted",
                                       style=STYLE, layout=wide("340px"))
trajectories = widgets.BoundedIntText(value=20, min=1, max=100000, description="max trajectories",
                                      style=STYLE, layout=wide("340px"))
workers = widgets.BoundedIntText(value=1, min=1, max=8, description="workers per GPU",
                                 style=STYLE, layout=wide("340px"))
write = widgets.Button(description="write settings file", button_style="primary",
                       layout=widgets.Layout(width="200px"))
written = widgets.Output()
CONFIG_PATH = None
SETTINGS = None

def on_type(change=None):
    modality = hpc.BINDER_TYPES[binder_type.value]
    scaffolded = modality in hpc.SCAFFOLDED_TYPES
    for box in (shortest, longest):
        box.layout.display = "none" if scaffolded else None
    copies.layout.display = None if modality == "homo_oligomer" else "none"
    preset = hpc.modality_lengths(modality)
    if preset and not scaffolded:
        shortest.value, longest.value = int(preset[0]), int(preset[1])

def on_write(_=None):
    global CONFIG_PATH, SETTINGS
    with written:
        written.clear_output(wait=True)
        if not TARGETS:
            print("add a target in cell 2 first")
            return
        try:
            SETTINGS = hpc.build_campaign_settings(
                campaign_name=campaign_name.value,
                targets=TARGETS,
                modality=hpc.BINDER_TYPES[binder_type.value],
                properties=[hpc.PROPERTY_OPTIONS[label] for label in properties.value],
                binder_lengths=(shortest.value, longest.value),
                number_of_final_designs=final_designs.value,
                max_trajectories=trajectories.value,
                workers_per_gpu=workers.value,
                copies=copies.value)
            CONFIG_PATH = hpc.write_campaign_config(SETTINGS)
        except Exception as failure:
            print(f"not written: {type(failure).__name__}: {failure}")
            return
        print(f"settings written to {CONFIG_PATH}\n")
        print(json.dumps(SETTINGS, indent=2))
        print("\nAcceptance filters come from the presets above and are left enabled.")

binder_type.observe(on_type, names="value")
write.on_click(on_write)
on_type()
display(widgets.VBox([campaign_name, binder_type, shortest, longest, copies, properties,
                      final_designs, trajectories, workers, write, written]))

## 4 - Submit the campaign to an H200 through SLURM

This calls `sbatch lab/slurm/run_bindcraft_h200.slurm <settings.json>`: partition `gpu`,
`--gres=gpu:h200:1`, 8 CPUs, 64 GB host RAM, 08:00:00. The launcher refuses to start if
`bindcraft` is imported from outside this source copy or if JAX cannot see a GPU, and it never
downloads weights. The job id is recorded under the results root, so a later notebook session
finds it again.

In [ ]:
account = widgets.Text(description="SLURM account (if any)", placeholder="leave empty unless your site requires one",
                       style=STYLE, layout=wide("720px"))
af2 = widgets.Text(description="AlphaFold params dir",
                   value=str(hpc.alphafold_parameters()[0] or ""),
                   placeholder="directory holding params_<model>.npz", style=STYLE, layout=wide("720px"))
dry_run = widgets.Checkbox(value=False, description="dry run (print the command, submit nothing)",
                           indent=False)
submit = widgets.Button(description="submit to H200", button_style="danger",
                        layout=widgets.Layout(width="200px"))
submitted = widgets.Output()
SUBMISSION = None

def on_submit(_=None):
    global SUBMISSION
    with submitted:
        submitted.clear_output(wait=True)
        if CONFIG_PATH is None:
            print("write a settings file in cell 3 first")
            return
        SUBMISSION = hpc.submit_campaign(CONFIG_PATH, account=account.value,
                                        af2_parameters=af2.value.strip(),
                                        dry_run=dry_run.value)
        print("command :", " ".join(SUBMISSION.command))
        print("message :", SUBMISSION.message)
        if SUBMISSION.job_id:
            print("job id  :", SUBMISSION.job_id)
            print("stdout  :", SUBMISSION.log_out)
            print("stderr  :", SUBMISSION.log_err)
            print("results :", SUBMISSION.project_folder)
            print("\nThe campaign now runs in the batch job. Closing this notebook does not stop it.")
        else:
            print("\nNothing was submitted. Nothing here claims a GPU ran.")

submit.on_click(on_submit)
display(widgets.VBox([account, af2, dry_run, submit, submitted]))

## 5 - Job status and log

`squeue` while the job is queued or running, `sacct` once it has finished. The log is the only
place that proves the campaign ran on a GPU: look for `xulab-bc2: jax ... backend=gpu` and
`xulab-bc2: device gpu:0 NVIDIA H200`.

In [ ]:
job_box = widgets.Text(description="job id (optional)", placeholder="leave empty for all recorded jobs",
                       style=STYLE, layout=wide())
lines = widgets.BoundedIntText(value=40, min=5, max=500, description="log lines", style=STYLE,
                               layout=wide("320px"))
refresh = widgets.Button(description="refresh status", button_style="primary",
                         layout=widgets.Layout(width="170px"))
show_log = widgets.Button(description="show log tail", layout=widgets.Layout(width="170px"))
cancel = widgets.Button(description="cancel this job", button_style="warning",
                        layout=widgets.Layout(width="170px"))
state_out = widgets.Output()

def current_job():
    typed = job_box.value.strip()
    if typed:
        return typed
    if SUBMISSION is not None and SUBMISSION.job_id:
        return SUBMISSION.job_id
    recorded = hpc.read_registry()
    return recorded[-1]["job_id"] if recorded else ""

def on_refresh(_=None):
    with state_out:
        state_out.clear_output(wait=True)
        records = hpc.tracked_jobs()
        if not records:
            print(f"no job recorded in {hpc.JOB_REGISTRY}")
            return
        for record in records:
            status = record.get("status", {})
            print(f"{record.get('job_id','?'):>10}  {record.get('campaign','?'):<28} "
                  f"{status.get('state','?'):<12} {status.get('elapsed','')} "
                  f"{status.get('exit_code', status.get('reason',''))}")
            print(f"            {record.get('project_folder','')}")

def on_log(_=None):
    with state_out:
        state_out.clear_output(wait=True)
        print(hpc.log_tail(current_job(), lines=lines.value))

def on_cancel(_=None):
    with state_out:
        state_out.clear_output(wait=True)
        job = current_job()
        print(hpc.cancel_job(job) if job else "no job id to cancel")

refresh.on_click(on_refresh)
show_log.on_click(on_log)
cancel.on_click(on_cancel)
display(widgets.VBox([job_box, lines, widgets.HBox([refresh, show_log, cancel]), state_out]))
on_refresh()

## 6 - Load a campaign and read its results

Works on a finished or a still-running campaign. With nothing accepted yet it shows the
candidates and the filters that rejected them, then the attempts and where they stopped - the
order the upstream output guide recommends.

In [ ]:
import pandas as pd   # tables only; the helper layer works without it

folders = hpc.campaign_folders()
picker = widgets.Dropdown(options=[folder.name for folder in folders] or ["(none found)"],
                          description="campaign", style=STYLE, layout=wide())
elsewhere = widgets.Text(description="or a full path", placeholder="/project/yli4/ypan/bindcraft/results/<name>",
                         style=STYLE, layout=wide("720px"))
rows = widgets.BoundedIntText(value=20, min=1, max=500, description="rows to show", style=STYLE,
                              layout=wide("320px"))
load = widgets.Button(description="load results", button_style="primary",
                      layout=widgets.Layout(width="170px"))
results_out = widgets.Output()
CAMPAIGN = None

def on_load(_=None):
    global CAMPAIGN
    with results_out:
        results_out.clear_output(wait=True)
        chosen_path = (Path(elsewhere.value.strip()) if elsewhere.value.strip()
                       else hpc.RESULTS_ROOT / picker.value)
        try:
            CAMPAIGN = hpc.load_campaign(chosen_path)
        except Exception as failure:
            print(f"{type(failure).__name__}: {failure}")
            return
        print(CAMPAIGN.summary())
        if CAMPAIGN.ranked:
            print("\nAccepted designs, best first:")
            display(hpc.ranked_view(CAMPAIGN, rows.value))
        if CAMPAIGN.candidates:
            print("\nScored candidates:")
            display(hpc.candidate_view(CAMPAIGN, rows.value))
            counts = hpc.rejection_counts(CAMPAIGN)
            if counts:
                print("rejected most often by:")
                for reason, count in list(counts.items())[:10]:
                    print(f"  {count:>5}  {reason}")
                print("('not measured' is a missing reading, not a low score)")
        if CAMPAIGN.attempts:
            print("\nTrajectory attempts (blank 'terminated' means gradient design completed):")
            display(hpc.attempt_view(CAMPAIGN, rows.value))
        if CAMPAIGN.metadata:
            keys = list(CAMPAIGN.metadata)[:10]
            print(f"\ncampaign_metadata.json keys: {keys}")

load.on_click(on_load)
display(widgets.VBox([picker, elsewhere, rows, load, results_out]))

## 7 - Interface information and confidence scores

Per accepted design: the sequence, the confidence readings (`i_pDAE`, `i_pTM`, `i_pAE`, `pLDDT`
and friends), the interface readings (contacting residues, buried area, hotspot contact,
clashes) and the structure files that belong to it. A design row with no structure file is
reported as such rather than hidden.

In [ ]:
if CAMPAIGN is None:
    print("load a campaign in cell 6 first")
elif not CAMPAIGN.ranked:
    print("nothing accepted yet, so there is no accepted-design report.")
    print("Cell 6 shows the candidates and the filters that rejected them.")
else:
    for entry in hpc.design_report(CAMPAIGN, limit=10):
        print("=" * 78)
        print(f"{entry['rank']:>3}  {entry['design']}")
        print(f"     chains     : {entry['chains']}")
        print(f"     sequence   : {entry['sequence']}")
        print(f"     structures : {', '.join(entry['structures'])}")
        if entry["confidence"]:
            print("     confidence : " + "  ".join(f"{k}={v}" for k, v in entry["confidence"].items()))
        if entry["interface"]:
            for key, value in entry["interface"].items():
                print(f"     {key:<34}{value}")
    print("=" * 78)
    print("i_pDAE / i_pTM / pLDDT / i_pAE are prediction confidence, not affinity.")
    print("Buried area is binder-side, in square angstrom; i_pAE is divided by 31 A.")

## 8 - 3D view of the predicted complexes

`py3Dmol` needs internet access for 3Dmol.js. On a closed network this cell says so and the
tables plus the CIF downloads in cell 9 remain the way to inspect the complexes.

In [ ]:
how_many = widgets.BoundedIntText(value=2, min=1, max=8, description="how many", style=STYLE,
                                  layout=wide("320px"))
colour_by = widgets.Dropdown(options=["chain", "pLDDT"], description="colour by", style=STYLE,
                             layout=wide("320px"))
draw = widgets.Button(description="show structures", button_style="primary",
                      layout=widgets.Layout(width="180px"))
view_out = widgets.Output()

def on_draw(_=None):
    with view_out:
        view_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        try:
            view, shown = hpc.show_structures(CAMPAIGN, how_many.value, colour_by.value)
        except RuntimeError as failure:
            print(failure)
            return
        for index, path in enumerate(shown, start=1):
            print(f"{index}. {path.name}  chains {','.join(hpc.chain_names(path)) or '?'}")
        view.show()

draw.on_click(on_draw)
display(widgets.VBox([how_many, colour_by, draw, view_out]))

## 9 - Export and download

Collects the binder FASTA, the ranked / candidate / attempt CSVs, `summary.csv`,
`campaign_metadata.json` and the accepted `.cif` complexes into one folder under
`results/exports/<campaign>/`, plus a zip. Nothing is written into the source checkout, so a
source update cannot delete an export.

In [ ]:
limit = widgets.BoundedIntText(value=20, min=1, max=500, description="designs to export",
                               style=STYLE, layout=wide("320px"))
run_export = widgets.Button(description="export results", button_style="primary",
                            layout=widgets.Layout(width="180px"))
export_out = widgets.Output()
EXPORT = None

def on_export(_=None):
    global EXPORT
    with export_out:
        export_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        EXPORT = hpc.export_campaign(CAMPAIGN, limit=limit.value)
        print(f"folder    : {EXPORT['folder']}")
        print(f"files     : {len(EXPORT['files'])}")
        print(f"sequences : {EXPORT['sequences']} FASTA record(s)")
        if EXPORT.get("zip"):
            print(f"zip       : {EXPORT['zip']}  ({EXPORT.get('zip_bytes', 0) / 1e6:.1f} MB)")
        for name in sorted(set(EXPORT["files"]))[:40]:
            print("   ", name)
        print("\nFASTA preview:")
        print(hpc.binder_fasta(CAMPAIGN, limit=3) or "(nothing accepted yet)")
        print("Download from the Jupyter file browser, or:")
        print(f"  scp <user>@<hpc-host>:{EXPORT.get('zip') or EXPORT['folder']} .")
        hpc.download_links([EXPORT.get("zip")] if EXPORT.get("zip") else [])

run_export.on_click(on_export)
display(widgets.VBox([limit, run_export, export_out]))

## 10 - Audit the campaign

Runs `lab/scripts/validate_outputs.py`, the same read-only validator used outside the notebook.
It grants **Level A (compute smoke)** only on explicit JAX GPU backend evidence plus a clean
campaign exit, and **Level B (end-to-end)** only when every accepted design has a sequence and a
structure. Zero accepted designs is reported as NOT ACHIEVED, which is not a failure.

In [ ]:
log_pattern = widgets.Text(description="log pattern",
                           value=str(hpc.LOGS_ROOT / "xulab_bc2_*.out"),
                           style=STYLE, layout=wide("720px"))
audit = widgets.Button(description="run validator", button_style="primary",
                       layout=widgets.Layout(width="180px"))
audit_out = widgets.Output()

def on_audit(_=None):
    with audit_out:
        audit_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        report_file = Path(EXPORT["folder"]) / "validation_report.md" if EXPORT else ""
        status, report = hpc.validate_campaign(CAMPAIGN.folder, log_glob=log_pattern.value,
                                               report_path=str(report_file) if report_file else "")
        print(report)
        print(f"validator exit status {status} "
              f"(0 = nothing failed, 1 = at least one FAIL, 2 = folder unreadable)")
        if report_file:
            print(f"report written to {report_file}")

audit.on_click(on_audit)
display(widgets.VBox([log_pattern, audit, audit_out]))

---

### Reading the outcome honestly

* **Zero accepted designs** after a short campaign is an ordinary result. Raise
  `max_trajectories` and resubmit; the campaign resumes into the same folder.
* **An accepted design is a computational result.** `i_pDAE`, `i_pTM` and pLDDT are prediction
  confidence, not affinity, specificity or stability. Nothing in this notebook is experimental
  evidence of binding.
* **GPU use is proven only by the job log.** This kernel never asserts that an H200 ran.
* **Campaign data lives under** `/project/yli4/ypan/bindcraft/results/`, outside the source
  checkout, so updating the code cannot delete it.

### Starting the first real campaign

```bash
# once, on a login node
mkdir -p /project/yli4/ypan/bindcraft/{results,logs,cache/jax,tmp}
cd /project/yli4/ypan/bindcraft/BindCraft2
bash lab/scripts/hpc_preflight.sh          # must print RESULT: READY

# then, in Jupyter
#   open lab/notebooks/xulab_bindcraft_h200.ipynb with the bindcraft2 kernel
#   cell 1 -> cell 2 (hPDL1) -> cell 3 (20 trajectories, 1 design) -> cell 4 submit
#   cell 5 to watch, cells 6-10 when it finishes
```

Upstream documentation: `docs/source/installation.md`, `docs/source/reference.md`,
`docs/source/outputs.md`. Xu Lab deployment notes: `README_XULAB.md`, `lab/README.md`.